# Lesson 10: 方言虚拟人搭建教程

本教程演示如何整合 GPT-SoVITS 语音合成和 Sadtalker 虚拟人技术，构建完整的方言虚拟人系统。

## 学习目标
- 理解语音合成和虚拟人技术
- 学习 GPT-SoVITS 零样本/少样本语音克隆
- 掌握 Sadtalker 音频驱动的面部动画
- 实现端到端的方言虚拟人系统

## 1. 环境准备

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, Video, display

from src.models.gpt_sovits_model import GPTSoVITSModel
from src.training.virtual_human_trainer import DialectVirtualHumanTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print("环境准备完成！")

## 2. 系统架构

### 完整流程
```
方言文本
    ↓
GPT-SoVITS 语音合成
    ↓
合成的方言语音
    ↓
Sadtalker 虚拟人生成
    ↓
虚拟人视频
```

### 核心技术

#### GPT-SoVITS
- **零样本 TTS**: 只需 5 秒参考音频即可克隆声音
- **少样本微调**: 1 分钟数据即可适应新说话人
- **多语言支持**: 支持中文、英文等多种语言

#### Sadtalker
- **音频驱动**: 根据音频生成面部动画
- **3D 感知**: 生成自然的头部运动
- **高质量**: 生成逼真的虚拟人视频

## 3. GPT-SoVITS 语音合成

### 零样本语音克隆
使用参考音频克隆声音，无需训练。

In [ ]:
# 配置
config = {
    'gpt_sovits_dir': './GPT-SoVITS',  # GPT-SoVITS 安装目录
    'gpt_model_path': 'pretrained_models/s1bert25hz-2kh-longer-epoch=68e-step=50232.ckpt',
    'sovits_model_path': 'pretrained_models/s2G488k.pth',
    'output_dir': 'results/tts'
}

# 初始化模型
tts_model = GPTSoVITSModel(config)

# 检查安装
if tts_model.check_installation():
    print("✓ GPT-SoVITS 已安装")
else:
    print("✗ GPT-SoVITS 未安装")
    print("\n安装步骤:")
    print("1. git clone https://github.com/RVC-Boss/GPT-SoVITS.git")
    print("2. cd GPT-SoVITS")
    print("3. pip install -r requirements.txt")
    print("4. 下载预训练模型")

## 4. 语音合成演示

In [ ]:
# 参考音频和文本
ref_audio = "data/reference/speaker1.wav"  # 参考音频（5秒以上）
ref_text = "这是参考文本"                  # 参考音频的文本

# 要合成的方言文本
dialect_texts = [
    "我今日好开心啊",
    "你食咗饭未啊",
    "呢度好靓啊"
]

print("语音合成配置:")
print(f"参考音频: {ref_audio}")
print(f"参考文本: {ref_text}")
print(f"\n要合成的文本:")
for i, text in enumerate(dialect_texts, 1):
    print(f"{i}. {text}")

# 合成语音（如果 GPT-SoVITS 已安装）
if Path(ref_audio).exists():
    print("\n开始合成...")
    
    for i, text in enumerate(dialect_texts):
        output_path = f"results/tts/output_{i}.wav"
        
        # 调用合成
        result = tts_model.synthesize(
            text=text,
            output_path=output_path,
            ref_audio=ref_audio,
            ref_text=ref_text
        )
        
        if result:
            print(f"✓ 合成完成: {output_path}")
            # 播放音频
            display(Audio(output_path))
        else:
            print(f"✗ 合成失败: {text}")
else:
    print(f"\n参考音频不存在: {ref_audio}")
    print("请准备参考音频后再运行此单元格")

## 5. 音频波形可视化

In [ ]:
def plot_waveform(audio_path, title="音频波形"):
    """绘制音频波形"""
    waveform, sample_rate = torchaudio.load(audio_path)
    waveform = waveform.numpy()
    
    time_axis = np.arange(waveform.shape[1]) / sample_rate
    
    plt.figure(figsize=(14, 4))
    plt.plot(time_axis, waveform[0], linewidth=0.5)
    plt.xlabel('时间 (秒)', fontsize=12)
    plt.ylabel('振幅', fontsize=12)
    plt.title(title, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"采样率: {sample_rate} Hz")
    print(f"时长: {waveform.shape[1] / sample_rate:.2f} 秒")

# 示例
audio_file = "results/tts/output_0.wav"
if Path(audio_file).exists():
    plot_waveform(audio_file, "合成语音波形")
else:
    print(f"音频文件不存在: {audio_file}")

## 6. Sadtalker 虚拟人生成

### 音频驱动的面部动画
使用合成的语音和人物图像生成虚拟人视频。

In [ ]:
# 虚拟人配置
vh_config = {
    'output_dir': 'results/virtual_human',
    'gpt_sovits': config,
    'sadtalker_dir': './SadTalker'  # Sadtalker 安装目录
}

# 初始化训练器
vh_trainer = DialectVirtualHumanTrainer(vh_config)

print("虚拟人训练器初始化完成！")
print("\nSadtalker 安装步骤:")
print("1. git clone https://github.com/OpenTalker/SadTalker.git")
print("2. cd SadTalker")
print("3. pip install -r requirements.txt")
print("4. 下载预训练模型")

## 7. 创建虚拟人视频

In [ ]:
# 虚拟人配置
avatar_image = "data/avatar/person1.jpg"  # 人物图像
dialect_text = "我今日好开心啊"          # 方言文本
output_video = "results/virtual_human/demo.mp4"

print("创建虚拟人视频:")
print(f"人物图像: {avatar_image}")
print(f"方言文本: {dialect_text}")
print(f"输出视频: {output_video}")

# 创建虚拟人（如果环境已配置）
if Path(avatar_image).exists() and Path(ref_audio).exists():
    print("\n开始创建虚拟人...")
    
    success = vh_trainer.create_dialect_virtual_human(
        dialect_text=dialect_text,
        ref_audio=ref_audio,
        ref_text=ref_text,
        avatar_image=avatar_image,
        output_video_path=output_video
    )
    
    if success:
        print(f"\n✓ 虚拟人创建成功: {output_video}")
        # 播放视频
        display(Video(output_video, width=640))
    else:
        print("\n✗ 虚拟人创建失败")
else:
    print(f"\n文件不存在，请准备:")
    print(f"  - 人物图像: {avatar_image}")
    print(f"  - 参考音频: {ref_audio}")

## 8. 批量创建虚拟人

In [ ]:
# 批量文本
batch_texts = [
    "我今日好开心啊",
    "你食咗饭未啊",
    "呢度好靓啊",
    "我哋去边度玩",
    "听日见"
]

output_dir = "results/virtual_humans_batch"

print(f"批量创建 {len(batch_texts)} 个虚拟人视频")
print(f"输出目录: {output_dir}")

# 批量创建（如果环境已配置）
if Path(avatar_image).exists() and Path(ref_audio).exists():
    print("\n开始批量创建...")
    
    results = vh_trainer.batch_create_virtual_humans(
        texts=batch_texts,
        ref_audio=ref_audio,
        ref_text=ref_text,
        avatar_image=avatar_image,
        output_dir=output_dir
    )
    
    success_count = sum(results)
    print(f"\n完成: {success_count}/{len(batch_texts)} 个视频")
    
    # 列出生成的视频
    if success_count > 0:
        print("\n生成的视频:")
        for i, (text, success) in enumerate(zip(batch_texts, results)):
            status = "✓" if success else "✗"
            print(f"{status} {i+1}. {text}")
else:
    print("\n环境未配置，跳过批量创建")

## 9. 使用 CLI 脚本

In [ ]:
cli_examples = """
# 1. 合成语音
python scripts/lesson_10_virtual_human.py --mode synthesize \\
    --text "我今日好开心啊" \\
    --ref_audio data/ref.wav \\
    --ref_text "参考文本" \\
    --output results/synthesized.wav

# 2. 创建虚拟人视频
python scripts/lesson_10_virtual_human.py --mode create_video \\
    --text "我今日好开心啊" \\
    --ref_audio data/ref.wav \\
    --ref_text "参考文本" \\
    --avatar_image data/avatar.jpg \\
    --output results/virtual_human.mp4

# 3. 批量创建虚拟人
python scripts/lesson_10_virtual_human.py --mode batch \\
    --input_file data/texts.txt \\
    --ref_audio data/ref.wav \\
    --ref_text "参考文本" \\
    --avatar_image data/avatar.jpg \\
    --output_dir results/virtual_humans

# 4. 训练 TTS 模型（可选）
python scripts/lesson_10_virtual_human.py --mode train \\
    --train_data_dir data/dialect_corpus \\
    --output_dir checkpoints/tts_model \\
    --epochs 10
"""

print("CLI 使用示例:")
print(cli_examples)

## 10. 性能优化建议

### GPU 加速
- 使用 GPU 可以大幅提升合成速度
- 推荐显存: 8GB+

### 批量处理
- 批量合成可以提高效率
- 建议批次大小: 4-8

### 模型选择
- 零样本: 快速，但质量可能略低
- 少样本微调: 质量更高，需要额外训练

### 质量 vs 速度
- 高质量: 使用更大的模型，增加采样步数
- 高速度: 使用较小的模型，减少采样步数

## 11. 应用场景

### 1. 方言教学
- 生成方言教学视频
- 互动式方言学习

### 2. 文化传承
- 保护濒危方言
- 数字化方言资源

### 3. 虚拟主播
- 方言新闻播报
- 方言直播

### 4. 游戏和娱乐
- 方言配音
- 虚拟角色

### 5. 无障碍服务
- 为不同方言用户提供服务
- 方言语音助手

## 12. 总结

本教程演示了：
1. ✅ GPT-SoVITS 零样本语音克隆
2. ✅ Sadtalker 虚拟人生成
3. ✅ 端到端的方言虚拟人系统
4. ✅ 批量处理流程
5. ✅ 实际应用场景

### 关键技术点
- **零样本 TTS**: 5 秒参考音频即可克隆声音
- **音频驱动**: 根据语音生成自然的面部动画
- **端到端**: 从文本到视频的完整流程
- **高质量**: 生成逼真的虚拟人

### 完整项目回顾

恭喜！你已经完成了所有 10 个课程模块：

1. ✅ **Lesson 1-3**: 理论基础（语音合成、MFA、Praat）
2. ✅ **Lesson 4**: SVM 元音分类
3. ✅ **Lesson 5**: MFA 模型训练
4. ✅ **Lesson 6**: LSTM 声调分类
5. ✅ **Lesson 7**: wav2vec IPA 识别
6. ✅ **Lesson 8**: 方言翻译（LoRA）
7. ✅ **Lesson 9**: 方言口音识别
8. ✅ **Lesson 10**: 方言虚拟人

### 下一步
- 在真实数据上训练模型
- 优化模型性能
- 部署到生产环境
- 探索更多应用场景

### 资源链接
- [GPT-SoVITS GitHub](https://github.com/RVC-Boss/GPT-SoVITS)
- [Sadtalker GitHub](https://github.com/OpenTalker/SadTalker)
- [项目文档](../README.md)

---

**感谢学习！祝你在方言语音处理领域取得成功！** 🎉